In [17]:
import numpy as np
from scipy.stats import norm
from matplotlib import pyplot as plt

T = 2
SIGMA = 0.25
R = 0.04
K = 200

t = np.arange(0.2, T + 0.2, 0.2)

def confidence_interval(s, l , N, level = 0.05):
    """Compute confidence interval"""
    z_alpha = norm.ppf(1 - level)
    return (l - z_alpha*s / np.sqrt(N), l + z_alpha*s / np.sqrt(N) )

def brownian_motion(t, mu = 0, sigma  = 1):
    """Standard brownian motion with drift mu and volatility sigma
    - Implemented according to Algorithm 5.17
    """

    # Sample a Wiener Process at times in t
    diff_t = np.append(np.zeros(1), t)
    diff_t = np.diff(diff_t)

    W = np.random.normal(0, np.sqrt(diff_t))

    # Add drift and Volatility
    B = np.cumsum(mu*diff_t + sigma*W)
    return B


def stock_price(X):
    """Implementation of stock price"""

    S_i = lambda i: 5 + i
    S = np.array([S_i(i) for i in range(1, 10 + 1)])
    S_hat = np.sum(S*np.exp(X), axis = 1)
    return S_hat


def crude_montecarlo(N , K):
    """Implementation of crude montecarlo algorithm
    Simulates the price of PI

     mu and sigma are parameters for Brownian motion
     """
    # Simulate 10 independent browninan motions and extract the
    # value at striketime T
    # X = np.array([brownian_motion(t,
    #                     mu = mu,
    #                     sigma = sigma
    #                     )
    #               for _ in range(10)])
    # X = X[:,-1]
    # X *= SIGMA
    # X += -T*SIGMA**2/2

    # Speed up using
    X = np.random.normal(-T*SIGMA**2/2, SIGMA*np.sqrt(T), size = (N, 10))

    S_hat = stock_price(X)

    PI = S_hat  - np.exp(-R*T)*K
    PI = np.maximum(PI, 0)

    s_PI = np.std(PI)
    PI = np.mean(PI)
    ci = confidence_interval(s_PI, PI, N)
    print(f"Crude Monte Carlo estimate of PI: {PI}")
    print(f"95% confidence interval: {ci}")
    print(f"Width of CI is {ci[1] - ci[0]}")
    print(f"Ratio of estimate to CI length: ", PI / (ci[1]- ci[0]))

    return PI

crude_montecarlo(N = 10000000, K = K)

Crude Monte Carlo estimate of PI: 1.638629786105652e-05
95% confidence interval: (np.float64(8.792904267773336e-06), np.float64(2.3979691454339703e-05))
Width of CI is 1.5186787186566367e-05
Ratio of estimate to CI length:  1.0789838337598616


np.float64(1.638629786105652e-05)

The event is so rare that, that for crude monetecarlo, $N = 10000$ produces a mean of 0.0.

Since $e^{X_i}$ are lognormal distributed, we can use the linearity of expectation to compute a mean drift such that
$$
E[\sum_i S_i(0) e^{X_i}] = 105E[e^{X_i}] \approx 185
$$
We know that $E[e^{X_i}] = e^{-T\sigma^2/2 + T\sigma^2/2} = e^0 =  1$, so we need to add a drift of $\mu$ such that
$$105*e^\mu = 185 \iff \mu = log(185/100)$$


In [21]:
drift = np.log(185/105)
print("Drift value: ", drift)
def importance_sampling_mc(N, K, drift):
    """Implementation of importance sampling algorithm"""

    def likilihood_ratio(X, drift):
        """
        Compute p(X) / q(X), where:
        Under P: X_i ~ N(m_P, variance)
        Under Q: X_i ~ N(m_P + drift, variance)
        """
        mean_p = -T*SIGMA**2/2
        mean_q = -T*SIGMA**2/2 + drift
        variance = SIGMA**2*T
        exponent = -((X - mean_p)**2 - (X - mean_q)**2)/(2*variance)
        return np.exp(exponent)

    # Sample from new measure
    X = np.random.normal(-T*SIGMA**2/2 + drift,
                         SIGMA*np.sqrt(T),
                         size = (N, 10))
    #Compute likelihood ratio, each row is sample, so need to compute the product each row since X = (X1,X2,..., X10) X_i iid
    L = np.prod(likilihood_ratio(X, drift), axis = 1)

    # Compute estimate weight by importance sampling likilhood ratio
    S_hat = stock_price(X)
    S_hat = S_hat
    payoff = np.maximum(S_hat - np.exp(-R*T)*K, 0)
    # Apply weighting and compute mean estimate.
    s_PI = np.std(payoff*L)

    PI = np.mean(payoff*L)
    ci = confidence_interval(s_PI, PI, N)
    print(f"Crude Monte Carlo estimate of PI: {PI}")
    print(f"95% confidence interval: {ci}")
    print(f"Width of CI is {ci[1] - ci[0]}")
    print(f"Ratio of estimate to CI length: ", PI / (ci[1]- ci[0]))
    return PI, L

l, weights = importance_sampling_mc(1000000, K , drift= drift)

Drift value:  0.5663954749208014
Crude Monte Carlo estimate of PI: 1.0125127443738062e-05
95% confidence interval: (np.float64(8.885600811266398e-06), np.float64(1.1364654076209725e-05))
Width of CI is 2.4790532649433266e-06
Ratio of estimate to CI length:  4.084271841561069
